# Lecture 4 · Notebook 3 — Inside a pretrained Transformer: attention maps, masks and positions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture04/lecture04_3_inspecting_attention.ipynb)

**Companion to Chapters [7](https://nlp-course-cuhksz.github.io/lecture04.html#p7-s86) and [8](https://nlp-course-cuhksz.github.io/lecture04.html#p8-s95) of the video lecture.** The attention weights on the slides were
illustrative. Here we measure real ones, in two small pretrained models:

* **DistilBERT** (`distilbert-base-uncased`, 6 layers × 12 heads) — an *encoder*: every token sees both sides;
* **GPT-2** (`gpt2`, 12 layers × 12 heads) — a *decoder*: every token sees only the past.

You will check the masks (slides 86–87), look at where *who* attends in the slides' sentence, find heads with recognisable jobs, look
at GPT-2's *learned* position embeddings, and time the quadratic cost of attention (slide 114).

Runs on CPU; downloading the two models (about 0.8 GB) takes most of the ~5 minutes. A GPU is not needed.

In [ ]:
!pip -q install transformers

In [ ]:
import time, numpy as np, torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

torch.set_grad_enabled(False)
# attn_implementation="eager" makes the models return their attention weights
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert-base-uncased", attn_implementation="eager").eval()
gpt_tok = AutoTokenizer.from_pretrained("gpt2")
gpt = AutoModel.from_pretrained("gpt2", attn_implementation="eager").eval()

def attentions(model, tok, text):
    enc = tok(text, return_tensors="pt")
    out = model(**enc, output_attentions=True)
    A = torch.stack(out.attentions)[:, 0]          # [layers, heads, query, key]
    return A, tok.convert_ids_to_tokens(enc["input_ids"][0]), enc

sentence = "Nobel committee awards Strickland who advanced optics."
A_bert, tb, enc_b = attentions(bert, bert_tok, sentence)
A_gpt, tg, enc_g = attentions(gpt, gpt_tok, sentence)
print("DistilBERT:", tuple(A_bert.shape), tb)
print("GPT-2     :", tuple(A_gpt.shape), tg)
print("every row sums to 1:", torch.allclose(A_bert.sum(-1), torch.ones(1)), torch.allclose(A_gpt.sum(-1), torch.ones(1)))

## 1 · Encoder vs. decoder masks (slides 86–87)
GPT-2 must not see the future, so its weights above the diagonal should be **exactly zero**. DistilBERT reads the whole input.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (A, toks, t) in zip(ax, [(A_bert, tb, "DistilBERT (encoder), mean of all heads"), (A_gpt, tg, "GPT-2 (decoder), mean of all heads")]):
    a.imshow(A.mean((0, 1)), cmap="Purples"); a.set_title(t)
    a.set_xticks(range(len(toks))); a.set_xticklabels(toks, rotation=60, ha="right")
    a.set_yticks(range(len(toks))); a.set_yticklabels(toks)
plt.tight_layout(); plt.show()
future = torch.triu(torch.ones(len(tg), len(tg), dtype=torch.bool), diagonal=1)
print("largest GPT-2 weight on a future token:", A_gpt[..., future].max().item())
print("largest DistilBERT weight on a future token:", round(A_bert[..., torch.triu(torch.ones(len(tb), len(tb), dtype=torch.bool), 1)].max().item(), 3))

Notice also how much weight goes to special tokens — `[CLS]`/`[SEP]` in DistilBERT, the first token in GPT-2. Many heads use them as a
"no-op" place to park attention when they have nothing useful to fetch.

## 2 · Where does *who* look?
On the slides (77, 97), *who* puts most of its weight on *Strickland*, the person it refers to. Those weights were illustrative.
Let us measure, for every DistilBERT layer and head, how much weight *who* puts on the word *Strickland* (summing over its word
pieces).

In [ ]:
words = enc_b.word_ids(0)                               # word index of every token (None for [CLS]/[SEP])
w_tokens = sentence.rstrip(".").split()
who_tok = [i for i, w in enumerate(words) if w == w_tokens.index("who")][0]
strick = [i for i, w in enumerate(words) if w == w_tokens.index("Strickland")]
print("tokens of 'Strickland':", [tb[i] for i in strick])

to_strick = A_bert[:, :, who_tok, strick].sum(-1)      # [layers, heads]
plt.figure(figsize=(8, 3.2)); plt.imshow(to_strick, cmap="Purples", vmin=0, vmax=1); plt.colorbar()
plt.xlabel("head"); plt.ylabel("layer"); plt.title("attention from 'who' to 'Strickland'"); plt.show()

L_, H_ = divmod(int(to_strick.argmax()), to_strick.shape[1])
print(f"strongest: layer {L_}, head {H_}: {to_strick[L_, H_]:.2f}   |   average over all heads: {to_strick.mean():.2f}")
row = A_bert[L_, H_, who_tok]
print("that head's full row for 'who':", {t: round(v, 2) for t, v in zip(tb, row.tolist())})

A few heads link *who* to *Strickland* strongly; on average a head gives it much less. But be careful: in this sentence *Strickland*
is also the word **right before** *who*. Is the strongest head resolving the reference, or just looking one word back? A control
sentence separates the two: here *who* still refers to Strickland (the physicist), but the word before it is *Canada*.

In [ ]:
def word_weights(text, layer, head, query_word):
    enc = bert_tok(text, return_tensors="pt")
    A = torch.stack(bert(**enc, output_attentions=True).attentions)[layer, 0, head]
    toks, wid = bert_tok.convert_ids_to_tokens(enc["input_ids"][0]), enc.word_ids(0)
    words = {}
    for t, w in zip(toks, wid):
        if w is not None:
            words[w] = words.get(w, "") + t.replace("##", "")
    q = [i for i, w in enumerate(wid) if w is not None and words[w] == query_word][0]
    out = {}
    for i, w in enumerate(wid):
        if w is not None:
            out[words[w]] = out.get(words[w], 0) + A[q, i].item()
    return out

control = "Nobel committee awards Strickland, a physicist from Canada who advanced optics."
for txt in [sentence, control]:
    ww = word_weights(txt, L_, H_, "who")
    top = sorted(ww.items(), key=lambda kv: -kv[1])[:4]
    print(f"layer {L_} head {H_} | {txt}")
    print("   'who' attends to: " + ", ".join(f"{w} {v:.2f}" for w, v in top))

In the control sentence the same head puts most of its weight on *Canada* and *from*: it is a **look-back head** that reads the
preceding words, and the slides' sentence just happens to put *Strickland* there. No head was told to resolve references —
whatever the heads do, they learned it from pre-training alone, and different heads learn different jobs (slide 103). Attention
patterns are easy to over-interpret; always test a story with a control.

## 3 · Heads with recognisable jobs
Two simple, measurable patterns in GPT-2:
* **previous-token heads** put their weight on the token just before the query;
* **first-token heads** park almost all weight on the first token.

We score every head on a longer text.

In [ ]:
text = ("The Transformer replaced recurrence with attention. Every token can look at every earlier token in one step, "
        "and the feed-forward network then transforms each position on its own. Stacking such blocks gives GPT-2.")
A_long, toks_long, _ = attentions(gpt, gpt_tok, text)
n = A_long.shape[-1]
prev = A_long[..., torch.arange(1, n), torch.arange(0, n - 1)].mean(-1)     # weight on the previous token
first = A_long[..., 1:, 0].mean(-1)                                          # weight on the first token

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (M, t) in zip(ax, [(prev, "previous-token score"), (first, "first-token score")]):
    im = a.imshow(M, cmap="Purples", vmin=0, vmax=1); a.set_title(t); a.set_xlabel("head"); a.set_ylabel("layer")
    plt.colorbar(im, ax=a)
plt.tight_layout(); plt.show()
l, h = divmod(int(prev.argmax()), 12)
print(f"most previous-token-like head: layer {l} head {h} (score {prev[l, h]:.2f})")
plt.figure(figsize=(5, 5)); plt.imshow(A_long[l, h, :30, :30], cmap="Purples"); plt.title(f"GPT-2 layer {l} head {h}: first 30 tokens")
plt.show()

## 4 · GPT-2's *learned* position embeddings
The original Transformer used fixed sinusoids (Notebook 2). GPT-2 instead **learns** one 768-dimensional vector for each of its 1,024
positions. What did it learn? Compare every pair of positions by cosine similarity.

In [ ]:
P = gpt.wpe.weight                                      # [1024, 768]
Pn = P / P.norm(dim=-1, keepdim=True)
sim = (Pn @ Pn.T)[:256, :256]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
im = ax[0].imshow(sim, cmap="viridis"); ax[0].set_title("cosine similarity of GPT-2 position embeddings"); plt.colorbar(im, ax=ax[0])
ax[0].set_xlabel("position"); ax[0].set_ylabel("position")
U, S, Vt = torch.pca_lowrank(P[:512], q=3)
xy = (P[:512] - P[:512].mean(0)) @ Vt[:, :2]
sc = ax[1].scatter(xy[:, 0], xy[:, 1], c=np.arange(512), cmap="plasma", s=6); plt.colorbar(sc, ax=ax[1], label="position")
ax[1].set_title("first two principal components (positions 0-511)")
plt.tight_layout(); plt.show()

Nobody told GPT-2 that position 100 is close to position 101, yet nearby positions end up with similar vectors and the positions
trace a smooth curve — learned from next-word prediction alone.

## 5 · The quadratic cost of attention (slide 114)
A sequence of n tokens creates an n × n attention matrix in every head of every layer. Watch memory and time as n grows.

In [ ]:
gpt(torch.randint(0, gpt_tok.vocab_size, (1, 16)))     # warm-up
for n in [128, 256, 512, 1024]:
    ids = torch.randint(0, gpt_tok.vocab_size, (1, n))
    t0 = time.time(); out = gpt(ids, output_attentions=True); dt = time.time() - t0
    attn_mb = sum(a.numel() for a in out.attentions) * 4 / 2**20
    print(f"n = {n:5d}:  attention weights = 12 layers x 12 heads x {n} x {n} = {attn_mb:7.1f} MB,  forward pass {dt:5.2f} s")

Doubling n multiplies the attention memory by four. The time grows more slowly at first, because for short inputs the projections and
feed-forward layers dominate the cost (slide 117). For GPT-2's 1,024 tokens this is still fine, but for 100k-token contexts it is
why sparse, compressed and linear attention variants exist (Chapter 9; slides 115–116), and why kernels such as FlashAttention avoid
storing the full matrix at all.

## Exercises
1. Try a sentence where the pronoun is ambiguous, e.g. *"The trophy didn't fit in the suitcase because it was too big."* Which noun
   does *it* attend to in the head you found in §2? Change *big* to *small* — does the pattern change?
2. Use `word_weights` on the control sentence for **every** layer and head. Is there a head whose weight from *who* goes mainly to
   *physicist* or *Strickland* rather than *Canada*? Does it pass the original sentence too?
3. Shuffle the words of a sentence and feed it to GPT-2. Does the previous-token head still attend to the previous *position*?
   What does that tell you about where that head gets its information?

**Back to the [video lecture]({SITE}#p8-s95)**